## 块
为什么会有块的出现，块和之前所接触的层之间有什么关系呢？  
* 神经网络-->层-->块  
他们都有着共同的特征，有输入、有输出、有参数，但他们是层层递进的关系，越往上越复杂，自然，功能也就强大  
神经网络块（block）可以描述单个层、由多个层组成的组件或者整个模型本身  
* 为什么需要使用块呢  
块与块之间可以组成更大的组件，实现更强大的功能  
***简单来说，块就是一个具有”输入、处理、输出能力的黑盒子。块最迷人的地方在于他的递归性，一个块可以由若干个层组成，甚至也可以是一个具有复杂结构的网络本身，你想想，能把如此多个复杂的东西结合在一起，最终能实现多么强大的功能啊，每一步都可以站在巨人的肩膀上“***
![多个层被组合成块，形成更大的模型](https://zh.d2l.ai/_images/blocks.svg)

## 块在编程中也由类（class）表示
和前面学过的神经网络类似，块也需要用类来表示   
这个类中必须包含   
* 将其输入转换为输出的前向传播函数
* 必要的参数（有些块是不需要含有任何函数的）   
* ***为了计算梯度，还必须要有反向传播函数***    
>但在pytorch中就很方便，他会根据 前向传播函数构建出一个计算图，自动推导出反向传播路径，因此我们不需要过多考虑

In [2]:
import torch
from torch import nn 
from torch.nn import functional as F

class MLP(nn.Module):
    '''__init__ 函数,定义网络的结构层次，只是准备好网络的层次结构，并没有真正的构建网络
    self.*表述对象本身的属性，*可以是任意合法的变量名'''
    def __init__(self):
        super().__init__()#调用父类的初始化函数__init__()，继承父类的属性
        self.hidden = nn.Linear(20, 256)#隐藏层
        self.out = nn.Linear(256, 10)#输出层
    '''forward 函数,定义网络的前向传播过程，输入数据经过网络层次结构的处理，得到输出结果'''
    def forward(self,X):
        return self.out(F.relu(self.hidden(X)))#前向传播过程，输入数据X经过隐藏层再通过激活函数，最后经过输出层得到最终的输出结果

In [3]:
X = torch.rand(2, 20)#生成一个2行20列的随机张量，作为输入数据
net = MLP()#创建一个MLP对象，实例化网络
net(X)

tensor([[ 2.0649e-02,  7.6609e-02,  2.8876e-01, -1.1843e-01,  1.2173e-01,
         -1.2231e-01,  7.0947e-02, -5.4388e-02,  1.3814e-01, -5.2813e-02],
        [-2.1262e-04,  3.0660e-02,  1.2151e-01,  1.4635e-02, -2.2155e-02,
         -9.4780e-02,  5.2126e-03, -8.8722e-02,  6.3534e-02, -7.4183e-02]],
       grad_fn=<AddmmBackward0>)

## 顺序块
***其实就是最开始学习的sequential函数***

In [5]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.0802,  0.2075, -0.1395, -0.0748, -0.2811,  0.2263, -0.1317,  0.1143,
         -0.1221,  0.0465],
        [-0.0326,  0.1796,  0.0103, -0.1480, -0.2617,  0.2276, -0.0540,  0.2414,
         -0.1014, -0.0382]], grad_fn=<AddmmBackward0>)

## 前向传播函数forward()中执行代码
***forward前向传播函数的本质也就只是一个普通的python函数，我们可以在其中运行任何逻辑，执行任意的数学运算，不止是简单的堆叠***

In [7]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)#生成一个20行20列的随机张量，作为固定权重，并且不需要计算梯度
        self.linear = nn.Linear(20, 20)#定义一个线性层，输入和输出都是20维
    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [8]:
net = FixedHiddenMLP()
net(X)

tensor(0.0991, grad_fn=<SumBackward0>)